# PEFT/LoRA 参数高效微调原理与实战

## LoRA 核心原理：低秩分解

**问题**：全量微调一个 7B 模型需要 ~28GB 显存（FP32），成本极高。

**LoRA 的核心思想**：预训练权重矩阵的变化量 ΔW 是**低秩**的，可以用两个小矩阵的乘积来近似。

### 数学表达

$$W' = W_0 + \Delta W = W_0 + BA$$

其中：
- $W_0 \in \mathbb{R}^{d \times d}$：预训练权重（冻结不动）
- $B \in \mathbb{R}^{d \times r}$：低秩矩阵
- $A \in \mathbb{R}^{r \times d}$：低秩矩阵
- $r \ll d$：秩（通常 r=8~64，而 d=4096~8192）

### 参数节省

```
全量微调参数：d × d = 4096 × 4096 = 16,777,216
LoRA 参数：  d × r + r × d = 2 × 4096 × 16 = 131,072
参数量缩减：128 倍！
```

### 直觉理解

想象你有一个训练好的大画师（预训练模型），你不需要让他重新学画画（全量微调），只需要教他一些新的"小技巧"（LoRA adapter）就能画出特定风格的作品（B站广告文案）。

In [ ]:
"""NumPy 演示：LoRA 低秩分解的参数节省效果"""
import numpy as np

# ============================================================
# 演示 LoRA 低秩分解：W' = W0 + B @ A
# ============================================================

d = 4096  # 典型的隐藏层维度（如 Llama-7B）
r = 16    # LoRA 秩

# 模拟原始权重矩阵（冻结）
np.random.seed(42)
W0 = np.random.randn(d, d).astype(np.float32) * 0.01

# LoRA 矩阵
B = np.random.randn(d, r).astype(np.float32) * 0.01  # d × r
A = np.random.randn(r, d).astype(np.float32) * 0.01  # r × d

# LoRA 更新：ΔW = B @ A
delta_W = B @ A  # 结果是 d × d 的矩阵

# 最终权重
W_new = W0 + delta_W

# 参数量对比
full_params = d * d
lora_params = d * r + r * d  # B + A

print("=" * 60)
print("LoRA 低秩分解参数量对比")
print("=" * 60)
print(f"隐藏维度 d = {d}, LoRA 秩 r = {r}")
print(f"W0 形状: {W0.shape} （冻结）")
print(f"B  形状: {B.shape}")
print(f"A  形状: {A.shape}")
print(f"ΔW 形状: {delta_W.shape}")
print()
print(f"全量微调参数量: {full_params:>15,} ({full_params * 4 / 1e6:.1f} MB, FP32)")
print(f"LoRA 微调参数量: {lora_params:>14,} ({lora_params * 4 / 1e6:.1f} MB, FP32)")
print(f"参数缩减比例:    {full_params / lora_params:.0f} 倍")

# ============================================================
# 不同模型规模的参数量对比
# ============================================================
print("\n" + "=" * 60)
print("不同模型规模的微调参数量对比（r=16）")
print("=" * 60)
print(f"{'模型':>10} {'总参数':>12} {'全量微调(FP32)':>16} {'LoRA(FP32)':>14} {'缩减':>6}")
print("-" * 60)

models = [
    ("1.5B", 1.5e9, 3072),
    ("7B", 7e9, 4096),
    ("13B", 13e9, 5120),
    ("70B", 70e9, 8192),
]

for name, total_params, hidden_dim in models:
    full_size_gb = total_params * 4 / 1e9  # FP32 = 4 bytes
    # LoRA 通常应用于 Q, K, V, O 四个矩阵
    num_layers = int(total_params / (12 * hidden_dim * hidden_dim))  # 近似层数
    lora_total = num_layers * 4 * 2 * hidden_dim * r  # 4个矩阵 × 2(B+A) × d × r
    lora_size_mb = lora_total * 4 / 1e6  # FP32
    print(f"{name:>10} {total_params/1e9:>10.1f}B {full_size_gb:>13.1f} GB {lora_size_mb:>12.1f} MB {full_size_gb*1000/lora_size_mb:>5.0f}x")

print("\n💡 关键发现：7B模型全量微调需要 ~28GB，LoRA 只需几 MB！")

In [ ]:
"""HuggingFace PEFT LoraConfig 配置实战"""

try:
    from peft import LoraConfig, get_peft_model, TaskType
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # ============================================================
    # LoRA 配置（B站广告文案生成场景）
    # ============================================================
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,                          # 秩：控制LoRA矩阵的大小
        lora_alpha=32,                 # 缩放系数：alpha/r = 缩放因子
        lora_dropout=0.05,             # Dropout 防止过拟合
        target_modules=[               # 要应用 LoRA 的模块
            "q_proj",                  # Query 投影
            "k_proj",                  # Key 投影
            "v_proj",                  # Value 投影
            "o_proj",                  # Output 投影
        ],
        bias="none",                   # 是否微调偏置
    )

    print("✅ LoRA 配置：")
    print(f"  秩 (r): {lora_config.r}")
    print(f"  Alpha: {lora_config.lora_alpha}")
    print(f"  缩放因子: {lora_config.lora_alpha / lora_config.r}")
    print(f"  Dropout: {lora_config.lora_dropout}")
    print(f"  目标模块: {lora_config.target_modules}")

    # 加载模型并应用 LoRA
    model_name = "Qwen/Qwen2-1.5B"
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
    peft_model = get_peft_model(model, lora_config)

    # 打印可训练参数统计
    peft_model.print_trainable_parameters()
    # 输出类似：trainable params: 4,194,304 || all params: 1,543,714,816 || trainable%: 0.27%

except ImportError as e:
    print(f"⚠️ 缺少依赖包：{e}")
    print("请安装：pip install peft transformers torch")
    print("\n模拟输出 LoRA 配置效果：")
    print("---")
    print("LoraConfig(")
    print("  task_type=CAUSAL_LM,")
    print("  r=16,                    # 秩")
    print("  lora_alpha=32,           # 缩放系数")
    print("  lora_dropout=0.05,       # Dropout")
    print("  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj']")
    print(")")
    print("\n预期效果（Qwen2-1.5B）：")
    print("  可训练参数: ~4M (0.27%)")
    print("  冻结参数:   ~1.5B (99.73%)")

## LoRA 超参数调优指南

### 关键超参数

| 超参数 | 推荐范围 | 说明 | B站文案场景建议 |
|--------|----------|------|----------------|
| **r（秩）** | 8-64 | 越大表达能力越强，但参数越多 | r=16（文案生成不需要太大秩） |
| **lora_alpha** | 16-32 | 缩放系数，通常设为 2r | alpha=32 |
| **lora_dropout** | 0.0-0.1 | 防过拟合，数据少时可调高 | 0.05 |
| **target_modules** | Q/K/V/O | 应用LoRA的注意力模块 | 至少包含 q_proj, v_proj |

### 秩 r 的选择经验

```
r=4-8   → 简单任务（分类、情感分析）
r=16-32 → 中等任务（文案生成、对话）      ← B站广告文案
r=64    → 复杂任务（代码生成、数学推理）
```

### target_modules 选择策略

- **最小配置**：`["q_proj", "v_proj"]` — 参数最少，效果基本够用
- **推荐配置**：`["q_proj", "k_proj", "v_proj", "o_proj"]` — 注意力全覆盖
- **最大配置**：再加上 `["gate_proj", "up_proj", "down_proj"]` — 包含FFN层

## 面试要点 🎯

### 1. 为什么 LoRA 能工作？

> **核心回答**：Aghajanyan et al. (2020) 的研究表明，预训练语言模型的权重更新矩阵具有很低的"内在维度"（intrinsic dimensionality）。换言之，微调时权重的变化是低秩的，不需要更新所有参数，只需要在低秩子空间中调整即可。

### 2. LoRA 的 r 怎么选？

> - 任务越复杂、与预训练数据差异越大 → r 越大
> - B站广告文案生成：中文文本生成任务，与预训练数据分布接近 → r=16 通常足够
> - 实际工作中：先用小 r 跑基线，再逐步增大观察效果是否提升

### 3. LoRA vs 全量微调的优劣？

| 维度 | LoRA | 全量微调 |
|------|------|----------|
| 显存占用 | 低（只存adapter梯度） | 高（所有参数梯度） |
| 训练速度 | 较快 | 较慢 |
| 效果上限 | 略低于全量微调 | 最佳 |
| 多任务部署 | 可共享基座+切换adapter | 每个任务一个完整模型 |
| 灾难性遗忘 | 较少（基座冻结） | 可能严重 |

### 4. LoRA 与 Adapter、Prefix-tuning 的区别？

> - **Adapter**：在每层之间插入小型网络，会增加推理延迟
> - **Prefix-tuning**：在输入前加可学习的"虚拟token"，占用上下文长度
> - **LoRA**：直接修改权重矩阵，推理时可合并，**零额外延迟** ← 这是LoRA最大的优势